# Schadenhöhe und reine Risikoprämie

Dieses Notebook prüft gespeicherte Entwicklungs- und Evaluationsergebnisse. Es passt kein Modell an, wählt kein Modell neu und verändert keine Zielgröße. Die Frequenzentscheidung stammt aus dem eingefrorenen Vergleich; die Schadenhöhenentscheidung wurde vor der gemeinsamen Evaluation gesperrt. Die frühere Exploration des vollständigen Datenstands bleibt offengelegt. Die Evaluation ist keine externe oder historisch vollkommen ungesehene Stichprobe.

[Registriertes Schadenhöhenprotokoll](../config/severity_protocol.json), [gesperrte Entwicklungsentscheidung](../reports/s5/development_decision.json). Die Quelle enthält historische französische EUR und erfasste positive Schadenzeilen; daraus folgt kein heutiger deutscher Marktpreis.

[Analytische Erläuterung und Ergebnisse](../docs/schadenhoehe_risikopraemie.md).

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image

start = Path.cwd().resolve()
ROOT = next(p for p in [start, *start.parents] if (p / 'config/severity_protocol.json').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data import sha256, text_sha256, json_bytes
from src.validation import read_artifacts, identity_id

reports = ROOT / 'reports/s5'
read_json = lambda path: json.loads(path.read_text(encoding='utf-8'))
development = read_json(reports / 'development_report.json')
evaluation = read_json(reports / 'evaluation_report.json')
decision = read_json(reports / 'development_decision.json')
release = read_json(reports / 'evaluation_release.json')
run_id, identity = development['run_id'], development['identity']
local = ROOT / 'data/severity' / run_id
assert run_id == identity_id('s5sev-', identity)
assert evaluation['run_id'] == run_id and evaluation['identity'] == identity
assert decision['identity'] == identity and decision['run_id'] == run_id
assert decision['decision_id'] == identity_id('s5decision-', {k: v for k, v in decision.items() if k != 'decision_id'})
assert release['evaluation_released'] is True
assert release['decision_id'] == evaluation['decision_id'] == decision['decision_id']
assert release['decision_sha256'] == evaluation['decision_sha256'] == sha256(reports / 'development_decision.json')
assert evaluation['release_sha256'] == sha256(reports / 'evaluation_release.json')
assert evaluation['selection_changed_after_evaluation'] is False
assert sha256(local / 'development/manifest.json') == decision['development_manifest_sha256']
assert sha256(local / 'registration.json') == development['registration_sha256']
assert sha256(local / 'development_decision.json') == sha256(reports / 'development_decision.json')
assert sha256(local.parent / 'evaluation_release.json') == evaluation['release_sha256']
assert identity['configuration_sha256'] == text_sha256(ROOT / 'config/severity_protocol.json')
for name, digest in identity['code_sha256'].items():
    assert text_sha256(ROOT / name) == digest
assert identity['input_snapshot_sha256'] == sha256(reports / 'input_snapshot.json')
assert identity['frequency_decision_sha256'] == sha256(ROOT / 'reports/s4/development_decision.json')
prepared = read_json(ROOT / 'reports/s1/preparation_report.json')
split_report = read_json(ROOT / 'reports/s2/split_manifest.json')
assert identity['source_id'] == prepared['source_id']
assert identity['preparation_id'] == prepared['preparation_id']
assert identity['split_id'] == split_report['split_id']
membership_tables, frozen = read_artifacts(ROOT / 'data/validation' / identity['split_id'])
assert frozen == split_report

tables = {}
for phase, report in [('development', development), ('evaluation', evaluation)]:
    tables[phase], saved = read_artifacts(local / phase)
    assert saved == report
    expected = membership_tables['policy_membership'].query('split == @phase').sort_values('IDpol')
    for name, frame in tables[phase].items():
        if name.endswith('predictions'):
            assert frame['split'].eq(phase).all()
            for model_id, part in frame.groupby('model_id'):
                ordered = part.sort_values('IDpol')
                assert ordered.IDpol.is_unique
                assert ordered.IDpol.tolist() == expected.IDpol.tolist()
                assert ordered.fold.tolist() == expected.fold.tolist()
        else:
            public = reports / f'{phase}_{name}.json'
            assert public.read_bytes() == json_bytes(json.loads(frame.to_json(orient='records', double_precision=15)))
for model_id, info in development['models'].items():
    assert read_json(local / model_id / 'manifest.json') == info
    assert sha256(local / model_id / 'predictions.parquet') == info['artifacts']['predictions']['sha256']
for model_id, info in development['development_refits'].items():
    assert read_json(local / f'{model_id}.refit.json') == info
    assert sha256(local / f'{model_id}.joblib') == info['sha256'] == decision['severity_refits'][model_id]

frequency = read_json(ROOT / 'reports/s4/frequency_report.json')
snapshot = read_json(reports / 'input_snapshot.json')
assert identity['frequency_run_id'] == frequency['run_id'] == snapshot['frequency_run_id']
frequency_local = ROOT / 'data/frequency' / frequency['run_id']
for model_id, info in frequency['models'].items():
    assert read_json(frequency_local / model_id / 'manifest.json') == info
    assert sha256(frequency_local / model_id / 'oof_predictions.parquet') == info['artifacts']['oof_predictions']['sha256']
for model_id, digest in decision['frequency_refits'].items():
    assert sha256(frequency_local / f'{model_id}.joblib') == digest

figure_manifest = read_json(reports / 'figures/manifest.json')
assert figure_manifest['run_id'] == run_id and figure_manifest['decision_id'] == decision['decision_id']
for name, digest in figure_manifest['source_sha256'].items():
    assert text_sha256(ROOT / name) == digest
for name, digest in figure_manifest['input_sha256'].items():
    assert sha256(ROOT / name) == digest
for info in figure_manifest['figures'].values():
    assert sha256(reports / 'figures' / info['filename']) == info['sha256']
selected = decision['selected_pp_model_id']
selected_severity = decision['selected_severity_model_id']
print('Lauf:', run_id, '| Entscheidung:', decision['decision_id'])
print('Gewähltes Produkt:', selected)
print('Quelle, Split, Code, Vorhersagen, Refits, Entscheidung, Freigabe, Tabellen und Abbildungen geprüft.')

## Positive Schadenzeilen: ein Gewicht je Claim

Die Schadenhöhe ist EUR je erfasster positiver Schadenzeile. Mehrere Schadenzeilen desselben Vertrags bleiben erhalten. Die mittlere Gamma-Deviance bewertet positive Schadenhöhen; MAE und RMSE sind zusätzlich in EUR je Schadenzeile angegeben. Keine dieser Größen verwendet Exponierungsgewichte.

Für die Claim-Kalibrierung werden Vorhersagen an tatsächlich gehaltenen Claims verglichen. Diese diagnostische Summe ist **keine** prospektive Verlustvorhersage für alle Verträge. Jeder spätere Vertrag erhält dieselbe Schadenhöhenregel unabhängig davon, ob er tatsächlich einen Schaden aufweist.

In [ ]:
GERMAN = {
    'model_id': 'Modell', 'split': 'Bestand', 'policies': 'Verträge',
    'policies_predicted': 'Verträge mit Vorhersage', 'claims': 'Positive Schadenzeilen',
    'observed_loss_eur': 'Beobachtete EUR', 'expected_loss_eur': 'Erwartete Perioden-EUR',
    'predicted_total_for_observed_claims_eur': 'Vorhersagesumme an beobachteten Schadenzeilen EUR',
    'observed_claim_mean_eur': 'Beobachtete EUR/Schaden',
    'predicted_claim_mean_eur': 'Vorhergesagte EUR/Schaden',
    'claim_actual_over_predicted': 'Schaden-A/P', 'mean_gamma_deviance': 'Mittlere Schaden-Gamma-Deviance',
    'claim_mae_eur': 'Schaden-MAE EUR', 'claim_rmse_eur': 'Schaden-RMSE EUR',
    'dimension': 'Merkmal', 'level': 'Ausprägung', 'supported_claims': 'Mindestens 100 Schadenzeilen',
    'supported_exposure': 'Mindestens 500 Vertragsjahre',
    'exposure_years': 'Vertragsjahre', 'zero_loss_policies': 'Verträge ohne Verlust',
    'actual_over_predicted': 'Verlust-A/P', 'tweedie_deviance_per_exposure': 'Gewichtete Jahresraten-Tweedie-Deviance',
    'mean_period_tweedie_deviance': 'Mittlere Perioden-Tweedie-Deviance',
    'observed_annual_loss_cost': 'Beobachtete EUR/Jahr', 'predicted_annual_loss_cost': 'Vorhergesagte EUR/Jahr',
    'loss_mae_period_eur': 'Perioden-MAE EUR', 'loss_rmse_period_eur': 'Perioden-RMSE EUR',
    'reference_model': 'Referenzmodell', 'deviance_difference': 'Score: Kandidat minus Referenz',
    'lower': 'Perzentil unten', 'upper': 'Perzentil oben',
    'actual_over_predicted_lower': 'A/P-Perzentil unten', 'actual_over_predicted_upper': 'A/P-Perzentil oben',
    'actual_over_predicted_difference': 'A/P-Differenz zur Referenz',
    'replicates': 'Ziehungen', 'fold': 'Falte', 'age_band': 'Fahreraltersgruppe',
    'mean_product_annual_pp': 'Mittel der Produkte EUR/Jahr',
    'product_of_separate_exposure_means': 'Produkt separater Mittel EUR/Jahr'
}
def show(frame):
    display(frame.rename(columns=GERMAN))

columns = ['model_id', 'split', 'policies_predicted', 'claims', 'observed_loss_eur',
           'predicted_total_for_observed_claims_eur', 'observed_claim_mean_eur',
           'predicted_claim_mean_eur', 'claim_actual_over_predicted',
           'mean_gamma_deviance', 'claim_mae_eur', 'claim_rmse_eur']
for phase in ['development', 'evaluation']:
    show(tables[phase]['severity_scores'][columns])
display(Image(filename=str(reports / 'figures/schadenhoehe_vergleich.png')))

## Claim-Kalibrierung und Unterstützung

Die Evaluation enthält Schäden aus zuvor gehaltenen Verträgen. Alters-, Bonus-Malus-, Kraftstoff- und Regionsgruppen zeigen Claim-Anzahl, Eurovolumen und mittlere Vorhersage. Die Grenze von 100 Schadenzeilen kennzeichnet Unterstützung; sie ist kein Signifikanztest und schützt nicht gegen den Einfluss einzelner großer Schäden. Ein gutes Gesamtergebnis garantiert keine gute Gruppenpassung.

In [ ]:
severity_focus = list(dict.fromkeys(['severity_mean', selected_severity]))
segments = tables['evaluation']['severity_segments']
for dimension in ['age_band', 'bm_band', 'VehGas', 'Region']:
    print('Merkmal:', dimension)
    show(segments.loc[segments.model_id.isin(severity_focus) & segments.dimension.eq(dimension),
         ['model_id', 'dimension', 'level', 'claims', 'observed_loss_eur',
          'observed_claim_mean_eur', 'predicted_claim_mean_eur',
          'claim_actual_over_predicted', 'supported_claims']])
print('Entwicklungs-Faltendiagnostik:')
show(tables['development']['severity_folds'][['model_id', 'fold', 'claims',
     'mean_gamma_deviance', 'claim_actual_over_predicted']])

## Alle Verträge: Jahresrisikoprämie und Periodenkosten

Für jeden Vertrag gilt `p_i = r_i × m_i` in EUR je Vertragsjahr und `c_i = Exposure_i × p_i` in EUR für die tatsächliche Vertragsperiode. Die Zielgröße enthält auch alle Nullverluste. Der Mittelwertzerlegung genügt beispielsweise `E[S|N,x,e] = N m(x)` zusammen mit `E[N|x,e] = e r(x)`; Unabhängigkeit der ganzen Verteilungen ist dafür nicht nötig. Der Datenabgleich belegt diese Annahmen nicht.

Der vorab feste Hauptscore ist `sum e_i d_1.5(S_i/e_i,p_i) / sum e_i`. Die separat gezeigte mittlere Perioden-Tweedie-Deviance ist bei wechselnder Exponierung **nicht identisch**. Beide sind Scores und keine Eurofehler. Die Wahl von Power 1,5 ist eine gemeinsame Scoringkonvention für Nullen und positive Verluste, keine Schätzung einer wahren Verteilung.

In [ ]:
columns = ['model_id', 'policies', 'zero_loss_policies', 'exposure_years',
           'observed_loss_eur', 'expected_loss_eur', 'observed_annual_loss_cost',
           'predicted_annual_loss_cost', 'actual_over_predicted',
           'tweedie_deviance_per_exposure', 'mean_period_tweedie_deviance',
           'loss_mae_period_eur', 'loss_rmse_period_eur']
for phase in ['development', 'evaluation']:
    print('Bestand:', phase)
    show(tables[phase]['loss_scores'][columns])
print('Gesperrte Entwicklungswahl:', selected, '|', decision['selection_rule'])
print('Gemeinsam freigegebene Frequenzmodelle:')
show(tables['evaluation']['frequency_scores'])
print('Vertrags- und Schadenbilanz Entwicklung:', development['reconciliation'])
print('Vertrags- und Schadenbilanz Evaluation:', evaluation['reconciliation'])

## Alter, Exponierung und Produktaggregation

Die Gruppenprofile verwenden den tatsächlich beobachteten Merkmalsmix jeder Altersgruppe. Sie sind keine standardisierten oder kausalen Alterskurven. Das Portfoliomittel ist `sum e_i r_i m_i / sum e_i`; ein Produkt getrennter Mittelwerte kann davon abweichen. Die beobachteten Verlustkosten bleiben der Vergleichsmaßstab.

Die Verlusttabellen kennzeichnen Unterstützung ab 500 Vertragsjahren. Besonders die Vertragsdauer bleibt eine wichtige Grenze: Eine gute globale A/P-Quote kann systematische Dauerabweichungen verdecken. Nutzung und Gründe des Beobachtungsendes sind nicht gemessen.

In [ ]:
segments = tables['evaluation']['loss_segments']
for dimension in ['age_band', 'bm_band', 'VehGas', 'Region', 'exposure_band', 'brand12']:
    print('Merkmal:', dimension, '| gewähltes Produkt:', selected)
    show(segments.loc[segments.model_id.eq(selected) & segments.dimension.eq(dimension),
         ['model_id', 'dimension', 'level', 'policies', 'exposure_years',
          'observed_loss_eur', 'expected_loss_eur', 'actual_over_predicted', 'supported_exposure']])
focus = list(dict.fromkeys([selected, 'pp_hgb_mean', 'pp_intercept_mean']))
components = tables['evaluation']['loss_components']
show(components.loc[components.model_id.isin(focus),
     ['model_id', 'age_band', 'policies', 'exposure_years', 'observed_annual_loss_cost',
      'mean_annual_frequency_exposure_weighted', 'mean_predicted_severity_policy_exposure_weighted',
      'mean_product_annual_pp', 'product_of_separate_exposure_means']])
display(Image(filename=str(reports / 'figures/alter_zerlegung.png')))

## Gepaarte Vertragsziehungen bei festen Vorhersagen

Die 500 Ziehungen verwenden ganze Verträge einschließlich Nullverlusten und halten damit alle zugehörigen Schadenzeilen zusammen. Alle Modelle erhalten dieselben gezogenen Indizes. Der Hauptscore wird je Ziehung durch ihre gezogene Exponierungssumme geteilt. Negative Score-Differenzen begünstigen den Kandidaten.

Die A/P-Perzentile beschreiben die Verlustkalibrierung bei festen Vorhersagen. Eine Differenz zweier A/P-Werte ist keine Differenz ihrer absoluten Kalibrierungsfehler. Die Intervalle enthalten kein erneutes Fitten, keine Modellwahlunsicherheit, keine unbekannte Haushaltsabhängigkeit und keine unbeobachteten Extremverluste. Die Entwicklungsintervalle sind nach Auswahl zu lesen; sie rechtfertigen keinen nachträglichen Signifikanztest.

In [ ]:
columns = ['model_id', 'reference_model', 'deviance_difference', 'lower', 'upper',
           'actual_over_predicted', 'actual_over_predicted_lower',
           'actual_over_predicted_upper', 'actual_over_predicted_difference',
           'actual_over_predicted_difference_lower',
           'actual_over_predicted_difference_upper', 'replicates', 'seed']
for phase in ['development', 'evaluation']:
    print('Bestand:', phase)
    show(tables[phase]['bootstrap'][columns])
display(Image(filename=str(reports / 'figures/risikopraemie_kalibrierung.png')))

## Oberer Schadenrand, Einfluss und Rücktransformation

Die Primäranalyse behält alle positiven Beträge ungekappt. Die Quantile und Verlustanteile beziehen sich auf tatsächliche Entwicklungs-Schadenzeilen. Eine deskriptive Kappung bei 25.000 EUR verändert das wirtschaftliche Ziel; sie wird nicht als Datenbereinigung übernommen.

Die Einflussanalyse entfernt beim Training alle Claims des Entwicklungsvertrags mit größter Verlustsumme. Sie vergleicht die Refits auf denselben Entwicklungs-Prognoseverträgen bei gleicher Frequenz. Dies ist eine Trainingssensitivität und keine neue unabhängige Evaluation.

Das Log-Modell verwendet `log(Y)`, nicht `log(1+Y)`. Seine EUR-Vorhersage ist `exp(eta) × mean_training(exp(log(Y)-eta))`. Globales Smearing ist nur eine Näherung an den bedingten Rücktransformationsfaktor. Gruppenunterschiede und große Residuen begrenzen die Interpretation; nach Einsicht in die Evaluation wird keine neue Smearingregel gewählt.

In [ ]:
tail = development['tail_diagnostics']
display(pd.DataFrame([{'Quantil': float(q), 'Tatsächlicher Schadenbetrag EUR': value}
                      for q, value in tail['quantiles_eur'].items()]).sort_values('Quantil'))
print('Anteil des größten Schadens:', tail['largest_claim_share'])
print('Anteil der größten aufgerundeten 1 % der Schadenzeilen:', tail['largest_ceil_one_percent_share'])
print('Hypothetische Kappung:', tail['descriptive_cap_eur'], 'EUR; entfernter Betrag:',
      tail['removed_eur_if_capped'], 'EUR; kein Primärausschluss.')
show(pd.DataFrame(tail['influence_refits']))
display(pd.DataFrame(tail['largest_claim_leverage']))
smear = development['development_refits']['severity_log_smear']['fit']
print('Globaler Trainings-Smearingfaktor:', smear['smearing_factor'])
print('Anteil des größten exp-Residuums:', smear['largest_exp_residual_share'])
display(pd.DataFrame(smear['smearing_groups']))
gamma = development['development_refits']['severity_gamma']['fit']
print('Gamma-Pearson-Diagnose:', gamma['pearson_dispersion'], '|', gamma['pearson_dispersion_scope'])
display(Image(filename=str(reports / 'figures/tail_einfluss.png')))

## Was die Ergebnisse tragen

Frequenz, Claim-Schadenhöhe und Verlustkosten bewerten verschiedene Größen. Eine bessere Claim-Deviance garantiert keine bessere Risikoprämie; ein niedriger Verlustscore garantiert keine passende Gesamtsumme oder Gruppenpassung. Ein geringer Nutzen zusätzlicher Schadenhöhenstruktur und breite Kalibrierungsintervalle sind zulässige Ergebnisse.

Die gesperrte Entscheidung bleibt nach der Evaluation unverändert. Weitere Modellverbesserungen würden neue Entwicklungsarbeit und eine ehrliche neue Validierungsfrage verlangen. Diese historische französische Studie liefert keinen unmittelbar verwendbaren deutschen Tarif.